## GPU ENGINE
[Rapids AI open beta Polars post](https://developer.nvidia.com/blog/polars-gpu-engine-powered-by-rapids-cudf-now-available-in-open-beta)

[Google Colab notebook demo from Rapids AI](https://colab.research.google.com/github/rapidsai-community/showcase/blob/main/accelerated_data_processing_examples/polars_gpu_engine_demo.ipynb?utm_source=nvidia+mktg&utm_medium=web&utm_campaign=polars+launch)

In [ ]:
import polars as pl
import os

zone_schema = {"LocationID": pl.Int32, "Borough": pl.String, "Zone": pl.String, "service_zone": pl.String}

df_zones = pl.scan_csv("data/nyc-taxi/taxi_zone_lookup.csv", schema=zone_schema)

df = (
    pl.scan_parquet("data/nyc-taxi/yellow.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)
df_green = (
    pl.scan_parquet("data/nyc-taxi/green.parquet")
    .join(df_zones, left_on="PULocationID", right_on="LocationID")
    .rename({"Borough": "PU_Borough", "Zone": "PU_Zone", "service_zone": "PU_service_zone"})
    .join(df_zones, left_on="DOLocationID", right_on="LocationID")
    .rename({"Borough": "DO_Borough", "Zone": "DO_Zone", "service_zone": "DO_service_zone"})
)

In [2]:
df.head().collect(engine="streaming")

VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,year,month,PU_Borough,PU_Zone,PU_service_zone,DO_Borough,DO_Zone,DO_service_zone
i16,datetime[ms],datetime[ms],u16,f32,u8,str,u16,u16,u8,f32,f32,f32,f32,f32,f32,f32,f32,f32,i16,i8,str,str,str,str,str,str
2,2014-01-04 19:23:00,2014-01-04 20:16:00,1,21.200001,1,null,132,188,1,60.5,0.0,0.5,10.0,0.0,0.0,71.0,null,null,2014,1,"""Queens""","""JFK Airport""","""Airports""","""Brooklyn""","""Prospect-Lefferts Gardens""","""Boro Zone"""
2,2014-01-04 19:50:00,2014-01-04 20:18:00,2,22.32,1,null,132,14,1,58.5,0.0,0.5,0.0,0.0,0.0,59.0,null,null,2014,1,"""Queens""","""JFK Airport""","""Airports""","""Brooklyn""","""Bay Ridge""","""Boro Zone"""
1,2014-01-04 19:14:12,2014-01-04 19:42:25,1,15.5,1,"""N""",132,150,1,43.0,0.0,0.5,5.0,0.0,0.0,48.5,null,null,2014,1,"""Queens""","""JFK Airport""","""Airports""","""Brooklyn""","""Manhattan Beach""","""Boro Zone"""
1,2014-01-04 19:04:06,2014-01-04 19:38:51,1,11.8,1,"""N""",132,82,2,35.0,0.0,0.5,0.0,0.0,0.0,35.5,null,null,2014,1,"""Queens""","""JFK Airport""","""Airports""","""Queens""","""Elmhurst""","""Boro Zone"""
1,2014-01-04 19:45:07,2014-01-04 20:04:04,1,3.7,1,"""N""",36,188,1,15.0,0.0,0.5,3.1,0.0,0.0,18.6,null,null,2014,1,"""Brooklyn""","""Bushwick North""","""Boro Zone""","""Brooklyn""","""Prospect-Lefferts Gardens""","""Boro Zone"""


## Streaming Engine

In [3]:
%%time
df_green.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine="streaming")

CPU times: user 7.31 s, sys: 122 ms, total: 7.44 s
Wall time: 342 ms


trip_distance,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,u32,f32,f32,f32,f32
-20329.080078,1,40.450001,40.450001,-20329.080078,0.0
-16183.30957,1,24.6,24.6,-16183.30957,0.0
-13844.679688,1,28.459999,28.459999,-13844.679688,0.0
-9436.330078,1,16.83,16.83,-9436.330078,0.0
-4311.350098,1,21.540001,21.540001,-4311.350098,0.0


## GPU in memory
Using the in-memory is the fastest than using managed memory (UVM streaming) but the memory is limited to GPU memory (as it Polars eager).

In [4]:
os.environ["POLARS_GPU_ENABLE_CUDA_MANAGED_MEMORY"] = "0"  # Forcing in-memory engine to not use UVM, just for the demo
gpu_in_mem_engine = pl.GPUEngine(executor="in-memory", raise_on_fail=True)

In [5]:
%%time
# This will fail if GPU memory is not large enough
df_green.group_by("trip_distance").agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort("trip_distance").head().collect(engine=gpu_in_mem_engine)

CPU times: user 1.77 s, sys: 267 ms, total: 2.03 s
Wall time: 581 ms


trip_distance,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,u32,f32,f32,f32,f32
-20329.080078,1,40.450001,40.450001,-20329.080078,0.0
-16183.30957,1,24.6,24.6,-16183.30957,0.0
-13844.679688,1,28.459999,28.459999,-13844.679688,0.0
-9436.330078,1,16.83,16.83,-9436.330078,0.0
-4311.350098,1,21.540001,21.540001,-4311.350098,0.0


In [6]:
# LazyFrame.profile is deprecated for the CPU engines from polars 1.43 (it assumes a single in-memory pass),
# but cudf-polars still documents it as supported on the GPU in-memory executor; streaming executors cannot profile.
result, prof = (
    df_green.group_by("trip_distance")
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort("trip_distance")
    .head()
    .profile(engine=gpu_in_mem_engine)
)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

node,start,end,pct_time_spent
str,u64,u64,f64
"""optimization""",0,161,0.082764
"""gpu-ir-translation""",161,682,0.267825
"""Scan""",775,100870,51.454788
"""Scan""",100885,102110,0.629723
"""Projection""",102111,102126,0.007711
…,…,…,…
"""Join""",128042,153354,13.011875
"""Projection""",153359,153368,0.004627
"""GroupBy""",153370,194058,20.916054


## GPU in memory. More complex queries

In [7]:
# Recommended to reset kernel to free GPU correctly because some part of the dataset is cached in memory from previous execution
# in the notebook and import again polars import and df definition at the top of the notebook

In [8]:
%%time
df_green.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine="streaming")

CPU times: user 20.1 s, sys: 360 ms, total: 20.5 s
Wall time: 938 ms


trip_distance,DO_Borough,PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,u32,f32,f32,f32,f32
-20329.080078,"""Brooklyn""","""Bronx""",1,40.450001,40.450001,-20329.080078,0.0
-16183.30957,"""Brooklyn""","""Brooklyn""",1,24.6,24.6,-16183.30957,0.0
-13844.679688,"""Bronx""","""Bronx""",1,28.459999,28.459999,-13844.679688,0.0
-9436.330078,"""Queens""","""Queens""",1,16.83,16.83,-9436.330078,0.0
-4311.350098,"""Queens""","""Queens""",1,21.540001,21.540001,-4311.350098,0.0


In [9]:
%%time
df_green.group_by(["trip_distance", "DO_Borough", "PU_Borough"]).agg(
    pl.len().alias("trip_count"),
    pl.sum("fare_amount").alias("total_revenue"),
    pl.mean("fare_amount").alias("avg_fare"),
    pl.mean("trip_distance").alias("avg_distance"),
    pl.mean("tip_amount").alias("avg_tip"),
).sort(["trip_distance", "DO_Borough", "PU_Borough"]).head().collect(engine=gpu_in_mem_engine)

CPU times: user 305 ms, sys: 46.7 ms, total: 352 ms
Wall time: 230 ms


trip_distance,DO_Borough,PU_Borough,trip_count,total_revenue,avg_fare,avg_distance,avg_tip
f32,str,str,u32,f32,f32,f32,f32
-20329.080078,"""Brooklyn""","""Bronx""",1,40.450001,40.450001,-20329.080078,0.0
-16183.30957,"""Brooklyn""","""Brooklyn""",1,24.6,24.6,-16183.30957,0.0
-13844.679688,"""Bronx""","""Bronx""",1,28.459999,28.459999,-13844.679688,0.0
-9436.330078,"""Queens""","""Queens""",1,16.83,16.83,-9436.330078,0.0
-4311.350098,"""Queens""","""Queens""",1,21.540001,21.540001,-4311.350098,0.0


In [10]:
result, prof = (
    df_green.group_by(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"])
    .agg(
        pl.len().alias("trip_count"),
        pl.sum("fare_amount").alias("total_revenue"),
        pl.mean("fare_amount").alias("avg_fare"),
        pl.mean("trip_distance").alias("avg_distance"),
        pl.mean("tip_amount").alias("avg_tip"),
    )
    .sort(["trip_distance", "DO_Zone", "DO_Borough", "PU_Borough"])
    .head()
    .profile(engine=gpu_in_mem_engine)
)
prof.with_columns(((pl.col("end") - pl.col("start")) / pl.col("end").max() * 100).alias("pct_time_spent"))

node,start,end,pct_time_spent
str,u64,u64,f64
"""optimization""",0,170,0.049814
"""gpu-ir-translation""",170,586,0.121897
"""Scan""",753,98718,28.705842
"""Scan""",98733,100345,0.472351
"""Projection""",100346,100361,0.004395
…,…,…,…
"""Join""",134530,194402,17.543777
"""Select""",194414,194433,0.005567
"""GroupBy""",194436,332781,40.538046
